# 01.无监督筛选

计算基础统计量、匹配率、同值率、PSI，并输出无监督筛选结果。

In [ ]:
import os
import sys
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "data" / "output"
SRC_DIR = PROJECT_ROOT / "src"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from 函数代码包 import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: "%.6f" % x)

customer_type = "demo_existing_customer"
target_type = "y2"   # 可切换为 "y1"

join_keys = ["id", "cell", "name", "apply_date"]
keep_cols = ["month", "flag"]
target = "y"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("customer_type:", customer_type)
print("target_type:", target_type)

In [ ]:
sample_file = DATA_DIR / f"sample_modeling_{target_type}.csv"
feature_file = DATA_DIR / "product_features.csv"

sample_df = read_csv_safely(sample_file)
feature_df = read_csv_safely(feature_file)

print(sample_df.shape, feature_df.shape)
display(sample_df.head())
display(feature_df.head())

In [ ]:
sample_df = sample_df.drop_duplicates(subset=join_keys, keep="first").copy()

data_df = pd.merge(
    sample_df,
    feature_df,
    on=join_keys,
    how="left"
)

print("merged shape:", data_df.shape)
display(data_df.head())

In [ ]:
exclude_cols = list(set(join_keys + keep_cols + [target, "y1", "y2"]))
var_cols = [c for c in data_df.columns if c not in exclude_cols]

stat_df = calc_basic_stat(data_df, var_cols)
stat_df.to_csv(OUTPUT_DIR / f"1.基础统计量_{customer_type}_{target_type}.csv", index=False)

display(stat_df.head())
print(stat_df["dtype"].value_counts())

In [ ]:
train_valid_df = data_df[data_df["flag"].isin(["train", "valid"])].copy()
oot_df = data_df[data_df["flag"] == "oot"].copy()

psi_df = calc_psi_by_var(train_valid_df, oot_df, var_cols, bins=10)
psi_df.to_csv(OUTPUT_DIR / f"1.PSI结果_train_valid_oot_{customer_type}_{target_type}.csv", index=False)

display(psi_df.head())

In [ ]:
unsup_result = stat_df.merge(
    psi_df[["var", "psi_max", "psi_avg"]],
    on="var",
    how="left"
)

unsup_result.to_csv(OUTPUT_DIR / f"1.无监督筛选结果_{customer_type}_{target_type}.csv", index=False)
data_df.to_csv(OUTPUT_DIR / f"1.建模宽表_{customer_type}_{target_type}.csv", index=False)

print("saved outputs.")
display(unsup_result.head())